In [ ]:


# %%
import numpy as np
import matplotlib.pyplot as plt
import pickle
from tqdm import tqdm
import os
import torch
import argparse
import yaml
#from numba import cuda
import zuko
from helpers.models.DNN import count_parameters
from helpers.data_transforms import (
    preprocess_data,
    inverse_preprocess_data,
    inverse_preprocess_data_torch,
    load_in_data,
)
from helpers.evaluation import get_kl_dist, discriminate_data_from_samples
from helpers.flow import sample_from_flow
plt.style.use("../science.mplstyle")
from helpers.material_map import torch_barrel_material_penalty

# %%
from helpers.plotting import plot_hists_1d, plot_corner_hist_2d

# %%


BIN_BOUND = 5
NUM_BINS = 100
NUM_FEATURES = 5

In [ ]:


feature_indices_dict = {
    "InnerTrackerBarrelCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
    "InnerTrackerEndcapCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
    "OuterTrackerBarrelCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
    "OuterTrackerEndcapCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
    "VertexBarrelCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
    "VertexEndcapCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
}


WORKING_DIR = "/pscratch/sd/r/rmastand/muon_collider"
ZUKO_ID = "NCSF"
NAME = "TEST"
SEED = 8
COLLECTION_LIST = "OuterTrackerBarrelCollection"
NUM_EPOCHS = 200
TRANSFORMS = 5 
FREQS = 5
HIDDEN_FEATURES = "128,128,128" 
TRAIN_FLOW = True  
EVAL_FLOW = True  
BATCH_SIZE = 4096 
LEARNING_RATE = 0.0001
NUM_COND_INPUTS = 2   
FEATURE_ORDER  = "0,4,1,2,3,6,7" 
BINS = 48  
DEGREE = 4
POLYNOMIALS = 5  
FEATURES = "rphi"
TRAINING_FRAC = 1
PHI_LOCAL = False
PLOT_EPOCH_INTERVAL = 1

lambda_material = 1 / BATCH_SIZE



In [ ]:




# %%
save_dir = f"{WORKING_DIR}/zuko_outputs/{ZUKO_ID}/{NAME}"
os.makedirs(save_dir, exist_ok=True)



# %%

# computing
device = torch.device( "cuda" if torch.cuda.is_available() else "cpu")
print( "Using device: " + str( device ), flush=True)
seed = int(SEED)
torch.manual_seed(seed)
np.random.seed(seed)

In [ ]:


collection_list = [x for x in COLLECTION_LIST.split(",")]
print(collection_list)


log_vars = []

# %%
FEATURE_ORDER = None if FEATURE_ORDER is None else [int(x) for x in FEATURE_ORDER.split(",")]
X, feature_labels = load_in_data(collection_list, FEATURES, WORKING_DIR, TRAINING_FRAC, NUM_COND_INPUTS, feature_order=FEATURE_ORDER, use_local_phi=PHI_LOCAL)
print(f"Data has shape {X.shape}")
print("Feature labels:", feature_labels)
NUM_FEATURES = X.shape[1] - NUM_COND_INPUTS


bins_dict = {}
bins_dict_preproc = {i:np.linspace(-BIN_BOUND, BIN_BOUND, NUM_BINS) for i in range(X.shape[1])}

for i in range(X.shape[1]):
    if i in log_vars:
        bins_dict[i] = np.logspace(np.log10(0.9*np.min(X[:,i])), np.log10(1.1*np.max(X[:,i])), NUM_BINS) 
    else:
        bins_dict[i] = np.linspace(np.min(X[:,i] - 1), np.max(X[:,i] + 1), NUM_BINS) 


fig_samp, axes_samp = plot_corner_hist_2d(
        X,
        feature_labels=feature_labels,
        bins_dict=bins_dict,
        log_dims=log_vars,
        title= "data",
    )
plt.savefig(f"{save_dir}/data_final")
plt.close()


In [ ]:

# %%


X_preproc = preprocess_data(X, save_dir, ZUKO_ID, NUM_COND_INPUTS)

# plot_hists_1d({"data":data}, bins_dict, log_dims=log_vars, labels=feature_labels)
# plt.show()

# plot_hists_1d({"data":X_preproc}, bins_dict_preproc, log_dims=[], labels = feature_labels)
# plt.show()





# %%
# train val split
from sklearn.model_selection import train_test_split


X_train, X_val = train_test_split(X_preproc, test_size=0.2, random_state=42)


print(f"Train data has shape {X_train.shape}.")
print(f"Val data has shape {X_val.shape}.")

train_loader = torch.utils.data.DataLoader(X_train, batch_size=BATCH_SIZE, shuffle=True, num_workers = 8, pin_memory = True)
val_loader = torch.utils.data.DataLoader(X_val, batch_size=BATCH_SIZE, shuffle=False, num_workers = 8, pin_memory = True)

# %%

In [ ]:



hidden_features = [int(x) for x in HIDDEN_FEATURES.split(",")]

if ZUKO_ID == "NSF":
    flow = zuko.flows.NSF(NUM_FEATURES, NUM_COND_INPUTS, transforms=TRANSFORMS, hidden_features=hidden_features).to(device)
#elif ZUKO_ID == "GMM":
#    flow = zuko.flows.GMM(NUM_FEATURES, NUM_COND_INPUTS, components=30, hidden_features=[256] * 5).to(device)
#elif ZUKO_ID == "NICE":
#    flow = zuko.flows.NICE(NUM_FEATURES, NUM_COND_INPUTS, transforms=TRANSFORMS, hidden_features=hidden_features).to(device)
elif ZUKO_ID == "MAF":
   flow = zuko.flows.MAF(NUM_FEATURES, NUM_COND_INPUTS, transforms=TRANSFORMS, hidden_features=hidden_features).to(device)
elif ZUKO_ID == "NCSF":
    flow = zuko.flows.NCSF(NUM_FEATURES, NUM_COND_INPUTS, transforms=TRANSFORMS, hidden_features=hidden_features, bins=BINS).to(device)
elif ZUKO_ID == "SOSPF":
    flow = zuko.flows.SOSPF(NUM_FEATURES, NUM_COND_INPUTS, transforms=TRANSFORMS, hidden_features=hidden_features, degree=DEGREE, polynomials=POLYNOMIALS).to(device)
#elif ZUKO_ID == "NAF":
#    flow = zuko.flows.NAF(NUM_FEATURES, NUM_COND_INPUTS, transforms=TRANSFORMS, hidden_features=hidden_features).to(device)
elif ZUKO_ID == "UNAF":
    flow = zuko.flows.UNAF(NUM_FEATURES, NUM_COND_INPUTS, transforms=TRANSFORMS, hidden_features=hidden_features).to(device)
elif ZUKO_ID == "CNF":
    flow = zuko.flows.CNF(NUM_FEATURES, NUM_COND_INPUTS, hidden_features=hidden_features, freqs=FREQS).to(device)
#elif ZUKO_ID == "GF":
#    flow = zuko.flows.GF(NUM_FEATURES, NUM_COND_INPUTS, transforms=TRANSFORMS, hidden_features=hidden_features, components=8).to(device)
#elif ZUKO_ID == "BPF":
#    flow = zuko.flows.BPF(NUM_FEATURES, NUM_COND_INPUTS, transforms=TRANSFORMS, hidden_features=hidden_features, degree=16).to(device)
else:
    print("ERROR: Unknown ZUKO_ID")
    exit()


num_params = count_parameters(flow)
print(f"Number of trainable parameters: {num_params}")


checkpoint_path = f"/pscratch/sd/r/rmastand/muon_collider/zuko_outputs/NCSF/TEST/latest_good_step11923.pt"

state_dict = torch.load(checkpoint_path, map_location=device)

flow.load_state_dict(state_dict)



In [ ]:
def run_training_step(data_loader, epoch, global_step, is_val_step=False):
    flow.eval() if is_val_step else flow.train()

    text_desc = "val" if is_val_step else "train"
    losses_ll, losses_mmap, losses_total = [], [], []
    bad_fracs_preproc = []
    bad_fracs_phys = []

    pbar = tqdm(data_loader, desc=f"{text_desc} batches", leave=False)

    for x in pbar:
        if not is_val_step:
            optimizer.zero_grad(set_to_none=True)

        x = x.to(device).float()

        if NUM_COND_INPUTS > 0:
            x_data = x[:, :-NUM_COND_INPUTS]
            x_context = x[:, -NUM_COND_INPUTS:]
            dist = flow(x_context)
        else:
            x_data = x
            x_context = None
            dist = flow()

        # -----------------------------
        # Check parameters before loss
        # -----------------------------
        bad_param = False

        for name, p in flow.named_parameters():
            if not torch.isfinite(p).all():
                print("\nNON-FINITE FLOW PARAMETER")
                print(name)
                print("finite frac:", torch.isfinite(p).float().mean().item())
                bad_param = True

        if bad_param:
            torch.save(
                {
                    "epoch": epoch,
                    "global_step": global_step,
                    "model_state_dict": flow.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict() if not is_val_step else None,
                },
                f"{save_dir}/nan_params_epoch{epoch}_step{global_step}.pt",
            )
            raise RuntimeError("Flow parameters became non-finite.")

        # -----------------------------
        # Log likelihood loss
        # -----------------------------
        log_prob = dist.log_prob(x_data)
        loss_ll = -log_prob.mean()

        if not torch.isfinite(loss_ll):
            debug_path = f"{save_dir}/nan_debug_epoch{epoch}_step{global_step}.pt"
            model_path = f"{save_dir}/nan_model_epoch{epoch}_step{global_step}.pt"

            bad_lp_mask = ~torch.isfinite(log_prob)
            bad_x_mask = ~torch.isfinite(x_data).all(dim=1)

            torch.save(
                {
                    "epoch": epoch,
                    "global_step": global_step,
                    "model_state_dict": flow.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict() if not is_val_step else None,
                },
                model_path,
            )

            torch.save(
                {
                    "epoch": epoch,
                    "global_step": global_step,
                    "loss_ll": loss_ll.detach().cpu(),
                    "log_prob": log_prob.detach().cpu(),
                    "bad_log_prob_mask": bad_lp_mask.detach().cpu(),
                    "bad_x_mask": bad_x_mask.detach().cpu(),
                    "x_full": x.detach().cpu(),
                    "x_data": x_data.detach().cpu(),
                    "x_context": None if x_context is None else x_context.detach().cpu(),
                    "bad_x_data": x_data[bad_lp_mask | bad_x_mask].detach().cpu(),
                    "bad_x_context": None if x_context is None else x_context[bad_lp_mask | bad_x_mask].detach().cpu(),
                },
                debug_path,
            )

            print("\nLL LOSS BECAME NON-FINITE")
            print("epoch:", epoch)
            print("global_step:", global_step)
            print("loss_ll:", loss_ll.item())
            print("num bad log_prob:", bad_lp_mask.sum().item(), "/", len(log_prob))
            print("num bad x_data:", bad_x_mask.sum().item(), "/", len(x_data))
            print("saved model:", model_path)
            print("saved debug batch:", debug_path)

            if bad_lp_mask.any():
                bad_idx = torch.where(bad_lp_mask)[0]
                print("first bad indices:", bad_idx[:20].detach().cpu().numpy())
                print("first bad x_data rows:")
                print(x_data[bad_idx[:5]].detach().cpu())
                if x_context is not None:
                    print("first bad x_context rows:")
                    print(x_context[bad_idx[:5]].detach().cpu())

            raise RuntimeError("Stopping because LL became NaN/inf.")

        # -----------------------------
        # Material loss
        # -----------------------------
        if not hasattr(dist, "rsample"):
            raise RuntimeError("dist does not support rsample(); material loss will not be differentiable.")

        rng_state = torch.get_rng_state()
        cuda_rng_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

        x_gen_preproc = dist.rsample()

        x_gen_preproc_clean = x_gen_preproc.clone()
        finite_mask_preproc = torch.isfinite(x_gen_preproc_clean).all(dim=1)
        x_gen_preproc_clean[~finite_mask_preproc] = x_gen_preproc[~finite_mask_preproc].detach() * 0.0
        
        # Build full tensor from the clean version
        if NUM_COND_INPUTS > 0:
            x_gen_preproc_full = torch.cat([x_gen_preproc_clean, x_context], dim=1)
        else:
            x_gen_preproc_full = x_gen_preproc_clean
        
        finite_mask = torch.isfinite(x_gen_preproc_full).all(dim=1)
        bad_frac_preproc = 1.0 - finite_mask.float().mean().item()
        bad_frac_phys = 0.0
        x_gen_phys_full = None
        finite_phys_mask = None


        x_gen_phys_full = None
        finite_phys_mask = None

        if finite_mask.any():
            x_gen_preproc_full_finite = x_gen_preproc_full[finite_mask]

            with torch.no_grad():

                x_gen_phys_full = inverse_preprocess_data_torch(
                    x_gen_preproc_full_finite,
                    save_dir,
                    ZUKO_ID,
                    NUM_COND_INPUTS,
                )

            finite_phys_mask = torch.isfinite(x_gen_phys_full).all(dim=1)
            bad_frac_phys = 1.0 - finite_phys_mask.float().mean().item()
            x_gen_phys = x_gen_phys_full[finite_phys_mask]

            if "Barrel" in collection_list[0] and len(x_gen_phys) > 0:
                material_loss = torch_barrel_material_penalty(
                    x_gen_phys,
                    collection_list[0],
                    feature_indices_dict,
                    softness=1.0,
                )
            else:
                material_loss = torch.zeros((), device=device)
        else:
            x_gen_phys = torch.empty((0, x_gen_preproc_full.shape[1]), device=device)
            material_loss = torch.zeros((), device=device)

        total_loss = loss_ll + lambda_material * material_loss

        # -----------------------------
        # Train step
        # -----------------------------
        if not is_val_step:
            # Save exact state before backward.
            # This is overwritten every step so it won't fill your filesystem.
            torch.save(
                {
                    "epoch": epoch,
                    "global_step": global_step,
                    "model_state_dict": flow.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "x": x.detach().cpu(),
                    "x_data": x_data.detach().cpu(),
                    "x_context": None if x_context is None else x_context.detach().cpu(),
                    "x_gen_preproc": x_gen_preproc.detach().cpu(),
                    "x_gen_preproc_full": x_gen_preproc_full.detach().cpu(),
                    "x_gen_phys_full": None if x_gen_phys_full is None else x_gen_phys_full.detach().cpu(),
                    "x_gen_phys": x_gen_phys.detach().cpu(),
                    "finite_mask": finite_mask.detach().cpu(),
                    "finite_phys_mask": None if finite_phys_mask is None else finite_phys_mask.detach().cpu(),
                    "log_prob": log_prob.detach().cpu(),
                    "loss_ll": loss_ll.detach().cpu(),
                    "material_loss": material_loss.detach().cpu(),
                    "total_loss": total_loss.detach().cpu(),
                    "lambda_material": lambda_material,
                    "rng_state": rng_state.cpu(),
                    "cuda_rng_state": cuda_rng_state,
                },
                f"{save_dir}/latest_before_backward.pt",
            )

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(flow.parameters(), 1.0, error_if_nonfinite=False)

            grad_summary = {}
            bad_grad = False
            bad_grad_name = None
            max_grad = 0.0

            for name, p in flow.named_parameters():
                if p.grad is None:
                    continue

                g = p.grad.detach()
                finite_g = torch.isfinite(g)

                grad_summary[name] = {
                    "shape": tuple(g.shape),
                    "finite_frac": finite_g.float().mean().item(),
                    "num_bad": (~finite_g).sum().item(),
                    "abs_max": torch.nan_to_num(
                        g, nan=0.0, posinf=0.0, neginf=0.0
                    ).abs().max().item(),
                    "mean": torch.nan_to_num(
                        g, nan=0.0, posinf=0.0, neginf=0.0
                    ).mean().item(),
                    "std": torch.nan_to_num(
                        g, nan=0.0, posinf=0.0, neginf=0.0
                    ).std().item(),
                }

                if not finite_g.all():
                    bad_grad = True
                    bad_grad_name = name
                    break

                max_grad = max(max_grad, g.abs().max().item())

            torch.save(
                {
                    "epoch": epoch,
                    "global_step": global_step,
                    "model_state_dict": flow.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "x": x.detach().cpu(),
                    "x_data": x_data.detach().cpu(),
                    "x_context": None if x_context is None else x_context.detach().cpu(),
                    "x_gen_preproc": x_gen_preproc.detach().cpu(),
                    "x_gen_preproc_full": x_gen_preproc_full.detach().cpu(),
                    "x_gen_phys_full": None if x_gen_phys_full is None else x_gen_phys_full.detach().cpu(),
                    "x_gen_phys": x_gen_phys.detach().cpu(),
                    "finite_mask": finite_mask.detach().cpu(),
                    "finite_phys_mask": None if finite_phys_mask is None else finite_phys_mask.detach().cpu(),
                    "log_prob": log_prob.detach().cpu(),
                    "loss_ll": loss_ll.detach().cpu(),
                    "material_loss": material_loss.detach().cpu(),
                    "total_loss": total_loss.detach().cpu(),
                    "lambda_material": lambda_material,
                    "rng_state": rng_state.cpu(),
                    "cuda_rng_state": cuda_rng_state,
                    "grad_summary": grad_summary,
                },
                f"{save_dir}/latest_after_backward_before_step.pt",
            )

            if bad_grad:
                print("\nNON-FINITE GRADIENT BEFORE OPTIMIZER STEP")
                print("param:", bad_grad_name)
                print("loss_ll:", loss_ll.item())
                print("material_loss:", material_loss.item())
                print("lambda_material:", lambda_material)
                print("total_loss:", total_loss.item())

                torch.save(
                    {
                        "epoch": epoch,
                        "global_step": global_step,
                        "model_state_dict": flow.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                    },
                    f"{save_dir}/bad_grad_model_epoch{epoch}_step{global_step}.pt",
                )

                torch.save(
                    {
                        "epoch": epoch,
                        "global_step": global_step,
                        "bad_grad_name": bad_grad_name,
                        "x": x.detach().cpu(),
                        "x_data": x_data.detach().cpu(),
                        "x_context": None if x_context is None else x_context.detach().cpu(),
                        "x_gen_preproc": x_gen_preproc.detach().cpu(),
                        "x_gen_preproc_full": x_gen_preproc_full.detach().cpu(),
                        "x_gen_phys_full": None if x_gen_phys_full is None else x_gen_phys_full.detach().cpu(),
                        "x_gen_phys": x_gen_phys.detach().cpu(),
                        "finite_mask": finite_mask.detach().cpu(),
                        "finite_phys_mask": None if finite_phys_mask is None else finite_phys_mask.detach().cpu(),
                        "log_prob": log_prob.detach().cpu(),
                        "loss_ll": loss_ll.detach().cpu(),
                        "material_loss": material_loss.detach().cpu(),
                        "total_loss": total_loss.detach().cpu(),
                        "lambda_material": lambda_material,
                        "rng_state": rng_state.cpu(),
                        "cuda_rng_state": cuda_rng_state,
                        "grad_summary": grad_summary,
                    },
                    f"{save_dir}/bad_grad_batch_epoch{epoch}_step{global_step}.pt",
                )

                raise RuntimeError("Stopping before optimizer step because gradient is non-finite.")

            

            
            optimizer.step()

            torch.save(
                {
                    "epoch": epoch,
                    "global_step": global_step,
                    "model_state_dict": flow.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "loss_ll": loss_ll.detach().cpu(),
                    "material_loss": material_loss.detach().cpu(),
                    "total_loss": total_loss.detach().cpu(),
                },
                f"{save_dir}/latest_after_step.pt",
            )

            global_step += 1




        losses_ll.append(loss_ll.item())
        losses_mmap.append(material_loss.item())
        losses_total.append(total_loss.item())
        bad_fracs_preproc.append(bad_frac_preproc)
        bad_fracs_phys.append(bad_frac_phys)

        pbar.set_postfix(loss=f"{total_loss.item():.3e}")

    metrics = {
        f"{text_desc}/ll": losses_ll,
        f"{text_desc}/mmap": losses_mmap,
        f"{text_desc}/total": losses_total,
        f"{text_desc}/bad_fracs_preproc": bad_fracs_preproc,
        f"{text_desc}/bad_fracs_phys": bad_fracs_phys,
    }

    return metrics, global_step

In [ ]:


if TRAIN_FLOW:
    print("Training flow...")

    # Train to maximize the log-likelihood
    optimizer = torch.optim.Adam(flow.parameters(), lr=LEARNING_RATE)

    # %%
    losses_train_ll, losses_train_mmap, losses_total = [], [], []
    bad_fracs_preproc, bad_fracs_phys = [], []
    best_val_loss = 1e10

    global_step = 11923

    for k in range(NUM_EPOCHS):

       

        train_losses, global_step = run_training_step(train_loader, k, global_step, is_val_step=False)

        torch.save(flow.state_dict(), f"{save_dir}/latest_good_step{global_step}.pt")
        losses_train_ll += train_losses["train/ll"]
        losses_train_mmap += train_losses["train/mmap"]
        losses_total += train_losses["train/total"]
        bad_fracs_preproc += train_losses["train/bad_fracs_preproc"]
        bad_fracs_phys += train_losses["train/bad_fracs_phys"]

        plt.figure()
        plt.plot(losses_train_ll)
        plt.ylabel("losses_train_ll")
        plt.title(k)
        plt.show()

        plt.figure()
        plt.plot(losses_train_mmap)
        plt.ylabel("losses_train_mmap")
        plt.title(k)
        plt.show()

        plt.figure()
        plt.plot(losses_total)
        plt.ylabel("losses_total")
        plt.title(k)
        plt.show()

        plt.figure()
        plt.plot(bad_fracs_preproc)
        plt.ylabel("bad_fracs_preproc")
        plt.title(k)
        plt.show()

        plt.figure()
        plt.plot(bad_fracs_phys)
        plt.ylabel("bad_fracs_phys")
        plt.title(k)
        plt.show()


        

        # with torch.no_grad():
        #     val_losses, _ = run_training_step(val_loader, k, global_step, is_val_step=True)


        
        #losses_train.append(train_losses["train/total"])
        # losses_val.append(val_losses["val/total"])
        
        # if val_losses["val/total"] < best_val_loss:
        #     best_val_loss = val_losses["val/total"]
        #     torch.save(flow.state_dict(), f"{save_dir}/test.pt")

        # if (k + 1) % PLOT_EPOCH_INTERVAL == 0:
        #     flow.eval()
        
        #     x_plot = next(iter(val_loader)).to(device).float()
        
        #     if NUM_COND_INPUTS > 0:
        #         x_plot_data = x_plot[:, :-NUM_COND_INPUTS]
        #         x_plot_context = x_plot[:, -NUM_COND_INPUTS:]
        #         factor = 5
        #         context_to_sample = x_plot_context.repeat_interleave(factor, dim=0)
        #         samples = sample_from_flow(flow, N=factor * len(x_plot_data), x_context=context_to_sample)
        #     else:
        #         factor = 5
        #         samples = sample_from_flow(flow, N=factor * len(x_plot))
        
        #     loc_data_dict = {
        #         # "data": inverse_preprocess_data(
        #         #     x_plot.detach().cpu().numpy(),
        #         #     save_dir,
        #         #     ZUKO_ID,
        #         #     NUM_COND_INPUTS,
        #         # ),
        #        "generated": inverse_preprocess_data(
        #             samples,
        #             save_dir,
        #             ZUKO_ID,
        #             NUM_COND_INPUTS,
        #         ),
        #     }
        #     plot_hists_1d(loc_data_dict, bins_dict, log_dims=log_vars, labels=feature_labels)
        #     plt.show()

        #     for key in loc_data_dict.keys():
        #         fig_samp, axes_samp = plot_corner_hist_2d(
        #             loc_data_dict[key],
        #             feature_labels=feature_labels,
        #             bins_dict=bins_dict,
        #             log_dims=log_vars,
        #             title= key,
        #         )
                
        #         plt.show()




In [1]:
import torch
import numpy as np
from pathlib import Path
from pprint import pprint

# -----------------------------
# Edit these
# -----------------------------
save_dir = Path("/pscratch/sd/r/rmastand/muon_collider/zuko_outputs/NCSF/OTBC_cond2_mmappenalty1/")

latest_after_step_path = save_dir / "latest_after_step.pt"
bad_batch_path = save_dir / "latest_after_backward_before_step.pt"
# or:
# bad_batch_path = save_dir / "bad_grad_batch_epochXX_stepYYYY.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Load checkpoints
# -----------------------------
good_ckpt = torch.load(latest_after_step_path, map_location="cpu")
bad_ckpt = torch.load(bad_batch_path, map_location="cpu")

print("Loaded good checkpoint:", latest_after_step_path)
print("  epoch:", good_ckpt["epoch"])
print("  global_step:", good_ckpt["global_step"])

print("\nLoaded bad batch:", bad_batch_path)
print("  epoch:", bad_ckpt["epoch"])
print("  global_step:", bad_ckpt["global_step"])
print("  bad_grad_name:", bad_ckpt.get("bad_grad_name", None))
print("  old loss_ll:", bad_ckpt["loss_ll"].item())
print("  old material_loss:", bad_ckpt["material_loss"].item())
print("  old total_loss:", bad_ckpt["total_loss"].item())

Loaded good checkpoint: /pscratch/sd/r/rmastand/muon_collider/zuko_outputs/NCSF/OTBC_cond2_mmappenalty1/latest_after_step.pt
  epoch: 16
  global_step: 10345

Loaded bad batch: /pscratch/sd/r/rmastand/muon_collider/zuko_outputs/NCSF/OTBC_cond2_mmappenalty1/latest_after_backward_before_step.pt
  epoch: 16
  global_step: 10346
  bad_grad_name: None
  old loss_ll: -1.4767122268676758
  old material_loss: 204.18287658691406
  old total_loss: -1.4268628358840942


In [2]:
import zuko

# Must match original run
ZUKO_ID = "NCSF"
NUM_FEATURES = 5
NUM_COND_INPUTS = 2
TRANSFORMS = 5
HIDDEN_FEATURES = [128,128,128]
BINS = 48
DEGREE = 4
POLYNOMIALS = 5
FREQS = 3
LEARNING_RATE = 1e-3

flow = zuko.flows.NCSF(
        NUM_FEATURES,
        NUM_COND_INPUTS,
        transforms=TRANSFORMS,
        hidden_features=HIDDEN_FEATURES,
        bins=BINS,
    ).to(device)


flow.load_state_dict(good_ckpt["model_state_dict"])

optimizer = torch.optim.Adam(flow.parameters(), lr=LEARNING_RATE)
optimizer.load_state_dict(good_ckpt["optimizer_state_dict"])

flow.train()

NCSF(
  (transform): LazyComposedTransform(
    (0): MaskedAutoregressiveTransform(
      (base): ComposedTransform(
        (0): CircularShiftTransform(bound=3.141592653589793)
        (1): MonotonicRQSTransform(bins=48)
      )
      (order): [0, 1, 2, 3, 4]
      (hyper): MaskedMLP(
        (0): MaskedLinear(in_features=7, out_features=128, bias=True)
        (1): ReLU()
        (2): MaskedLinear(in_features=128, out_features=128, bias=True)
        (3): ReLU()
        (4): MaskedLinear(in_features=128, out_features=128, bias=True)
        (5): ReLU()
        (6): MaskedLinear(in_features=128, out_features=715, bias=True)
      )
    )
    (1): MaskedAutoregressiveTransform(
      (base): ComposedTransform(
        (0): CircularShiftTransform(bound=3.141592653589793)
        (1): MonotonicRQSTransform(bins=48)
      )
      (order): [4, 3, 2, 1, 0]
      (hyper): MaskedMLP(
        (0): MaskedLinear(in_features=7, out_features=128, bias=True)
        (1): ReLU()
        (2): MaskedL

In [3]:
# -----------------------------
# Utilities
# -----------------------------
def grad_report(model):
    rows = []
    for name, p in model.named_parameters():
        if p.grad is None:
            continue

        g = p.grad.detach()
        finite = torch.isfinite(g)

        rows.append({
            "name": name,
            "shape": tuple(g.shape),
            "finite_frac": finite.float().mean().item(),
            "num_bad": (~finite).sum().item(),
            "abs_max": torch.nan_to_num(
                g, nan=0.0, posinf=0.0, neginf=0.0
            ).abs().max().item(),
            "mean": torch.nan_to_num(
                g, nan=0.0, posinf=0.0, neginf=0.0
            ).mean().item(),
            "std": torch.nan_to_num(
                g, nan=0.0, posinf=0.0, neginf=0.0
            ).std().item(),
        })

    rows_bad = [r for r in rows if r["num_bad"] > 0]
    rows_worst = sorted(rows, key=lambda r: r["abs_max"], reverse=True)

    return rows_bad, rows_worst


def print_grad_report(title, model):
    rows_bad, rows_worst = grad_report(model)

    print(f"\n========== {title} ==========")
    print("num params with bad grad:", len(rows_bad))

    if rows_bad:
        print("\nBad gradients:")
        for r in rows_bad[:20]:
            print(
                f"{r['name']:80s} "
                f"finite_frac={r['finite_frac']:.6f} "
                f"num_bad={r['num_bad']} "
                f"abs_max={r['abs_max']:.3e}"
            )

    print("\nLargest gradients:")
    for r in rows_worst[:20]:
        print(
            f"{r['name']:80s} "
            f"finite_frac={r['finite_frac']:.6f} "
            f"num_bad={r['num_bad']} "
            f"abs_max={r['abs_max']:.3e}"
        )


def tensor_report(name, x):
    if x is None:
        print(f"{name}: None")
        return

    finite = torch.isfinite(x)

    print(f"\n{name}")
    print("  shape:", tuple(x.shape))
    print("  dtype:", x.dtype)
    print("  finite frac:", finite.float().mean().item())
    print("  num bad:", (~finite).sum().item())

    if x.numel() > 0:
        x_clean = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        print("  min:", x_clean.min().item())
        print("  max:", x_clean.max().item())
        print("  mean:", x_clean.float().mean().item())
        print("  std:", x_clean.float().std().item())

In [4]:
# -----------------------------
# Re-run likelihood on bad batch
# -----------------------------
x = bad_ckpt["x"].to(device).float()

if NUM_COND_INPUTS > 0:
    x_data = x[:, :-NUM_COND_INPUTS]
    x_context = x[:, -NUM_COND_INPUTS:]
    dist = flow(x_context)
else:
    x_data = x
    x_context = None
    dist = flow()



In [5]:
from helpers.data_transforms import inverse_preprocess_data_torch
from helpers.material_map import torch_barrel_material_penalty

collection_name = "OuterTrackerBarrelCollection"

feature_indices_dict = {
    "InnerTrackerBarrelCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
    "InnerTrackerEndcapCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
    "OuterTrackerBarrelCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
    "OuterTrackerEndcapCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
    "VertexBarrelCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
    "VertexEndcapCollection": {"r": 2, "phi": 3, "z": 4, "side": 5, "layer": 6},
}

lambda_material = bad_ckpt["lambda_material"]

# Restore RNG from the bad checkpoint if available
if bad_ckpt.get("rng_state", None) is not None:
    torch.set_rng_state(bad_ckpt["rng_state"])

if torch.cuda.is_available() and bad_ckpt.get("cuda_rng_state", None) is not None:
    torch.cuda.set_rng_state_all(bad_ckpt["cuda_rng_state"])

# Rebuild dist after resetting RNG
if NUM_COND_INPUTS > 0:
    dist = flow(x_context)
else:
    dist = flow()

x_gen_preproc = dist.rsample()
x_gen_preproc = torch.nan_to_num(x_gen_preproc, nan=0.0, posinf=0.0, neginf=0.0)



if NUM_COND_INPUTS > 0:
    x_gen_preproc_full = torch.cat([x_gen_preproc, x_context], dim=1)
else:
    x_gen_preproc_full = x_gen_preproc

#finite_mask = torch.isfinite(x_gen_preproc_full).all(dim=1)

print("\nGenerated sample check:")
tensor_report("x_gen_preproc", x_gen_preproc.detach().cpu())
tensor_report("x_gen_preproc_full", x_gen_preproc_full.detach().cpu())
#print("finite generated rows:", finite_mask.sum().item(), "/", len(finite_mask))

x_gen_preproc_full_finite = x_gen_preproc_full#[finite_mask]


x_gen_phys_full = inverse_preprocess_data_torch(
    x_gen_preproc_full_finite,
    str(save_dir),
    ZUKO_ID,
    NUM_COND_INPUTS,
)

#finite_phys_mask = torch.isfinite(x_gen_phys_full).all(dim=1)
x_gen_phys = x_gen_phys_full#[finite_phys_mask]


print("\nPhysical sample check:")
tensor_report("x_gen_phys_full", x_gen_phys_full.detach().cpu())
tensor_report("x_gen_phys", x_gen_phys.detach().cpu())
#print("finite physical rows:", finite_phys_mask.sum().item(), "/", len(finite_phys_mask))

material_loss = torch_barrel_material_penalty(
    x_gen_phys,
    collection_name,
    feature_indices_dict,
    softness=1.0,
)
print(material_loss)

total_loss = lambda_material * material_loss

print("\nRecomputed losses:")
print("material_loss:", material_loss.item())
print("lambda_material:", lambda_material)
print("total_loss:", total_loss.item())

print_grad_report("LL + material backward from latest_after_step", flow)
print("**********")
x_gen_preproc.retain_grad()
x_gen_preproc_full.retain_grad()
x_gen_phys.retain_grad()

optimizer.zero_grad(set_to_none=True)
total_loss.backward(retain_graph=True)

for name, t in [
    ("x_gen_phys.grad", x_gen_phys.grad),
    ("x_gen_preproc_full.grad", x_gen_preproc_full.grad),
    ("x_gen_preproc.grad", x_gen_preproc.grad),
]:
    print("\n", name)
    print("is None:", t is None)
    if t is not None:
        print("finite frac:", torch.isfinite(t).float().mean().item())
        print("num bad:", (~torch.isfinite(t)).sum().item())
        print("min/max clean:",
              torch.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0).min().item(),
              torch.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0).max().item())
print_grad_report("LL + material backward from latest_after_step", flow)
torch.nn.utils.clip_grad_norm_(flow.parameters(), 1.0, error_if_nonfinite=False)



Generated sample check:

x_gen_preproc
  shape: (4096, 5)
  dtype: torch.float32
  finite frac: 1.0
  num bad: 0
  min: -3.1415865421295166
  max: 3.1415913105010986
  mean: -0.642988383769989
  std: 1.805040955543518

x_gen_preproc_full
  shape: (4096, 7)
  dtype: torch.float32
  finite frac: 1.0
  num bad: 0
  min: -3.1415865421295166
  max: 3.1415913105010986
  mean: -0.3400321304798126
  std: 1.642354965209961

Physical sample check:

x_gen_phys_full
  shape: (4096, 7)
  dtype: torch.float32
  finite frac: 1.0
  num bad: 0
  min: -1263.9384765625
  max: 1493.427978515625
  mean: 159.02847290039062
  std: 481.7233581542969

x_gen_phys
  shape: (4096, 7)
  dtype: torch.float32
  finite frac: 1.0
  num bad: 0
  min: -1263.9384765625
  max: 1493.427978515625
  mean: 159.02847290039062
  std: 481.7233581542969
tensor(204.2142, device='cuda:0', grad_fn=<MeanBackward0>)

Recomputed losses:
material_loss: 204.21420288085938
lambda_material: 0.000244140625
total_loss: 0.04985698312520981



tensor(nan, device='cuda:0')

In [ ]:
import torch
import numpy as np
from pathlib import Path
from pprint import pprint

save_dir = Path("/pscratch/sd/r/rmastand/muon_collider/zuko_outputs/NCSF/OTBC_cond2_mmappenalty1/")
ckpt_path = save_dir / "latest_after_backward_before_step.pt"

ckpt = torch.load(ckpt_path, map_location="cpu")

print("Loaded:", ckpt_path)
print("epoch:", ckpt["epoch"])
print("global_step:", ckpt["global_step"])
print("loss_ll:", ckpt["loss_ll"].item())
print("material_loss:", ckpt["material_loss"].item())
print("total_loss:", ckpt["total_loss"].item())
print("lambda_material:", ckpt["lambda_material"])
print()
print("keys:")
print(ckpt.keys())

In [ ]:
def tensor_report(name, x):
    if x is None:
        print(f"{name}: None")
        return

    finite = torch.isfinite(x)
    print(f"\n{name}")
    print("  shape:", tuple(x.shape))
    print("  dtype:", x.dtype)
    print("  finite frac:", finite.float().mean().item())
    print("  num bad:", (~finite).sum().item())

    if x.numel() > 0:
        x_clean = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        print("  min:", x_clean.min().item())
        print("  max:", x_clean.max().item())
        print("  mean:", x_clean.float().mean().item())
        print("  std:", x_clean.float().std().item())


for key in [
    "x",
    "x_data",
    "x_context",
    "x_gen_preproc",
    "x_gen_preproc_full",
    "x_gen_phys",
    "log_prob",
    "finite_mask",
    "finite_phys_mask",
]:
    tensor_report(key, ckpt.get(key))

In [ ]:
x_gen = ckpt["x_gen_preproc"]
bad_gen_mask = ~torch.isfinite(x_gen).all(dim=1)

print("bad generated preproc rows:", bad_gen_mask.sum().item(), "/", len(x_gen))

if bad_gen_mask.any():
    bad_idx = torch.where(bad_gen_mask)[0]
    print("first bad indices:", bad_idx[:20].numpy())
    print("first bad rows:")
    print(x_gen[bad_idx[:5]])

In [ ]:
grad_summary = ckpt["grad_summary"]

bad = []
for name, s in grad_summary.items():
    if s["num_bad"] > 0:
        bad.append((name, s))

print("num params with bad grad:", len(bad))

for name, s in bad[:20]:
    print("\nBAD GRAD:", name)
    pprint(s)

In [ ]:
worst = sorted(
    grad_summary.items(),
    key=lambda kv: kv[1]["abs_max"],
    reverse=True,
)

print("largest finite-ish gradients:")
for name, s in worst[:20]:
    print(f"{name:80s} abs_max={s['abs_max']:.3e} finite_frac={s['finite_frac']:.6f} num_bad={s['num_bad']}")